# Kaggle runner — exp_0013 / exp_0014 (FT-Transformer reproduction)

Trains the two step-8 reproduction experiments on GPU, on the **frozen 5-fold
partition** shipped as the `s6e7-frozen-folds` dataset (never rebuilt — rule 6):

- **exp_0013** — FT-Transformer (Kawamata recipe via `masamlp`), 13 raw features.
  One variable vs exp_0001: the model family.
- **exp_0014** — same + 39 per-value target-encoding features (`catstat`, fitted
  inside each fold). One variable vs exp_0013: the representation.

Ledger rows record **raw argmax** CV, like every model row; the prior-corrected
decision rule is measured at home as exp_0015/exp_0016 (`cv.run_rule`, cross-fitted).
Libraries pinned to the source notebook's versions. All logic imports from `src/` —
this notebook only orchestrates and displays.

**Carry back** (`kaggle kernels output`): `artifacts/exp_001{3,4}{,_test}.npy` →
`oof/`, and the two new rows of `artifacts/experiments.csv` → the local ledger.

In [ ]:
# Diagnostics first, stdlib only. Kaggle's log endpoint has returned empty files for
# errored runs, so this cell makes the run self-reporting: mounts go to an output file,
# and any later cell's exception is written to diag_error.txt before it propagates
# (output files survive an errored run).
import json
import sys
import traceback
from pathlib import Path

diag = {
    "python": sys.version,
    "inputs": {
        p.name: sorted(f.name for f in p.iterdir())[:10]
        for p in Path("/kaggle/input").iterdir()
    },
}
Path("/kaggle/working/diag_mounts.json").write_text(json.dumps(diag, indent=2))
print(json.dumps(diag, indent=2))


def _dump_exc(shell, etype, evalue, tb, tb_offset=None):
    text = "".join(traceback.format_exception(etype, evalue, tb))
    Path("/kaggle/working/diag_error.txt").write_text(text)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)


get_ipython().set_custom_exc((BaseException,), _dump_exc)

In [ ]:
# --no-deps is load-bearing: a plain install lets pip replace the image's torch with a
# PyPI wheel that lacks kernels for the assigned GPU's architecture (run 5 died with
# "no kernel image is available for execution on the device" on exactly that). Both
# libraries' dependencies (torch, numpy, pandas, scikit-learn) ship with the image.
%pip install -q --no-deps masamlp==0.3.0 catstat==0.4.0

In [ ]:
import shutil
import subprocess

# The repo lives in /tmp, NOT /kaggle/working: only artifacts/ and diag files should
# land in the output snapshot, so a failed run stays kilobytes and pulls in seconds.
REPO = Path("/tmp/repo")

clone = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/epsilonlog/comp-playground-series-s6e7.git", str(REPO)],
    capture_output=True, text=True,
)
assert (REPO / "src").exists(), f"clone failed: {clone.stderr}"

inputs = Path("/kaggle/input")


def first_existing(*candidates: Path) -> Path:
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"none exist: {[str(c) for c in candidates]}")


# Mount layout differs across Kaggle images: classic /kaggle/input/<slug> vs the
# nested /kaggle/input/competitions/<slug> and /kaggle/input/datasets/<owner>/<slug>
# (run 3's diag showed this image uses the nested form).
comp = first_existing(
    inputs / "competitions" / "playground-series-s6e7",
    inputs / "playground-series-s6e7",
)
ds = first_existing(
    inputs / "datasets" / "aligh474" / "s6e7-frozen-folds",
    inputs / "s6e7-frozen-folds",
)

raw = REPO / "data" / "raw"
processed = REPO / "data" / "processed"
raw.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)

for n in ["train.csv", "test.csv", "sample_submission.csv"]:
    src = comp / n if (comp / n).exists() else ds / n
    assert src.exists(), f"{n} found in neither {comp} nor {ds}"
    shutil.copy(src, raw / n)
shutil.copy(ds / "folds.parquet", processed / "folds.parquet")

print("csv source:", comp)
print("raw:", sorted(p.name for p in raw.iterdir()))
print("processed:", sorted(p.name for p in processed.iterdir()))

In [ ]:
import sys

sys.path.insert(0, "/tmp/repo/src")

import catstat
import masamlp
import numpy as np
import torch

from s6e7 import cv, decision, features, folds, io
from s6e7.cv import ExperimentConfig

print("masamlp", masamlp.__version__, "| catstat", catstat.__version__,
      "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())
train, test = io.load_train(), io.load_test()
folds.verify(train)  # the shipped file IS the frozen partition; prove it arrived intact
print("folds verified:", folds.FOLDS_PATH)

Smoke first: does each wrapper run end to end on GPU? 34k rows, 2 epochs, never
logged. The scores are meaningless **by design** — per-value TE cannot be screened at
small n (the source notebook measured it at −0.0017 on a 70k screen vs +0.0012 at
full scale). This cell only proves the plumbing before an hour of training.

In [ ]:
smoke = train.head(34_000)
for name in ("ftt", "ftt_te"):
    r = cv.run(
        ExperimentConfig(exp_id=f"smoke_{name}", model=name, params={"n_epochs": 2}),
        train=smoke,
        log=False,
    )
    print(f"{name}: plumbing ok, {r.runtime_s:.0f}s (scores meaningless at this n)")

In [ ]:
result_13 = cv.run(
    ExperimentConfig(
        exp_id="exp_0013",
        model="ftt",
        parent="exp_0001",
        changed="new family: FT-Transformer (masamlp, Kawamata recipe), raw features",
    ),
    train=train,
    test=test,
    if_logged="skip",
)
print(f"exp_0013  cv_mean={result_13.cv_mean:.5f}  cv_std={result_13.cv_std:.5f}  (raw argmax)")
print("fold scores:", [round(s, 5) for s in result_13.fold_scores])

In [ ]:
result_14 = cv.run(
    ExperimentConfig(
        exp_id="exp_0014",
        model="ftt_te",
        parent="exp_0013",
        changed="add 39 per-value target-encoding features (catstat, fitted inside the fold)",
    ),
    train=train,
    test=test,
    if_logged="skip",
)
print(f"exp_0014  cv_mean={result_14.cv_mean:.5f}  cv_std={result_14.cv_std:.5f}  (raw argmax)")
print("fold scores:", [round(s, 5) for s in result_14.fold_scores])

Preview of the number that actually matters — the prior-corrected score. In-sample
on the full OOF (fine as a preview: two parameters on 550k rows overfit by ≈0 — see
exp_0009's cache); the honest cross-fitted rows are logged at home as exp_0015/0016.

In [ ]:
y = features.encode_target(train[io.TARGET])
for exp_id in ("exp_0013", "exp_0014"):
    proba = np.load(REPO / "oof" / f"{exp_id}.npy")
    multipliers, score = decision.search(proba, y)
    print(f"{exp_id}: rule preview {score:.5f}  multipliers {np.round(multipliers, 3).tolist()}")

In [ ]:
art = Path("/kaggle/working/artifacts")
art.mkdir(exist_ok=True)
for rel in ["oof/exp_0013.npy", "oof/exp_0013_test.npy",
            "oof/exp_0014.npy", "oof/exp_0014_test.npy", "experiments.csv"]:
    shutil.copy(REPO / rel, art / Path(rel).name)
print(sorted(f"{p.name} ({p.stat().st_size:,}B)" for p in art.iterdir()))